In [ ]:
import pandas as pd
import pickle
import os
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.feature_selection import SelectFromModel
from sklearn import metrics
from sklearn import preprocessing
from sklearn.metrics import average_precision_score
from sklearn.metrics import precision_recall_curve
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from inspect import signature
from matplotlib.ticker import FormatStrFormatter
from sklearn.model_selection import train_test_split
import matplotlib.lines as mlines
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn import metrics
import seaborn as sns
from sklearn.metrics import average_precision_score
from matplotlib.lines import Line2D
import scipy.io as sio
from sklearn.impute import KNNImputer
from scipy import stats
from sklearn.utils import resample

from sklearn.model_selection import GridSearchCV

from sklearn.feature_selection import SelectKBest, f_classif
import pickle
from sklearn.linear_model import SGDClassifier
from matplotlib.colors import ListedColormap, LinearSegmentedColormap
import itertools

In [ ]:
def plot_confusion_matrix(cm,
                          target_names,
                          title='Confusion matrix',
                          cmap=None,
                          normalize=True):
    """
    given a sklearn confusion matrix (cm), make a nice plot

    Arguments
    ---------
    cm:           confusion matrix from sklearn.metrics.confusion_matrix

    target_names: given classification classes such as [0, 1, 2]
                  the class names, for example: ['high', 'medium', 'low']

    title:        the text to display at the top of the matrix

    cmap:         the gradient of the values displayed from matplotlib.pyplot.cm
                  see http://matplotlib.org/examples/color/colormaps_reference.html
                  plt.get_cmap('jet') or plt.cm.Blues

    normalize:    If False, plot the raw numbers
                  If True, plot the proportions

    Usage
    -----
    plot_confusion_matrix(cm           = cm,                  # confusion matrix created by
                                                              # sklearn.metrics.confusion_matrix
                          normalize    = True,                # show proportions
                          target_names = y_labels_vals,       # list of names of the classes
                          title        = best_estimator_name) # title of graph

    Citiation
    ---------
    http://scikit-learn.org/stable/auto_examples/model_selection/plot_confusion_matrix.html

    """
    import matplotlib.pyplot as plt
    import numpy as np
    import itertools

    accuracy = np.trace(cm) / float(np.sum(cm))
    misclass = 1 - accuracy

    if cmap is None:
        cmap = plt.get_cmap('Blues')
    
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        
    plt.figure(figsize=(8, 6),dpi=300)
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()

    if target_names is not None:
        tick_marks = np.arange(len(target_names))
        plt.xticks(tick_marks, target_names, rotation=45)
        plt.yticks(tick_marks, target_names)


    thresh = cm.max() / 1.5 if normalize else cm.max() / 2
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        if normalize:
            plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")
        else:
            plt.text(j, i, "{:,}".format(cm[i, j]),
                     horizontalalignment="center",
                     color="white" if cm[i, j] > thresh else "black")


    plt.tight_layout()
    plt.ylabel('True label')
    plt.xlabel('Predicted label\naccuracy={:0.4f}; misclass={:0.4f}'.format(accuracy, misclass))
    plt.show()

# Load Data

In [ ]:
#study_criteria_table = pd.read_excel(r"..\medical_data\study_criteria_table_label_V4.xlsx")
study_criteria_table = pd.read_excel(r"../medical_data/study_criteria_table_label_V6.xlsx")

In [ ]:
study_criteria_table

In [ ]:
criteria_df = study_criteria_table

In [ ]:
#study_features_table = pd.read_csv(r"${DEMENTIA_DATA_ROOT}\dementia_detection\eeg_data\study_features_table_v4.csv")
study_features_table = pd.read_csv(r"../eeg_data/study_features_table_v4.csv")

In [ ]:
study_features_table

# Preprocess Data

In [ ]:
data_df = study_criteria_table[['FolderName','Predicted_Stage']].merge(study_features_table,on=['FolderName'])

In [ ]:
for index,row in data_df.iterrows():
    if data_df.loc[[index]].isna().sum().sum() > 800:
        data_df = data_df.drop([index])
    

In [ ]:
for index,row in data_df[data_df['Predicted_Stage'] == 'Dementia'].iterrows():
    print('i',index)
    print(data_df.loc[[index]].isna().sum().sum())

    

In [ ]:
d = data_df[data_df['Predicted_Stage'] == 'Dementia']

In [ ]:
data_df = data_df.drop_duplicates(subset=['FolderName'])

In [ ]:
dementia_df = pd.DataFrame(data=criteria_df[(criteria_df['Predicted_Stage']== 'Dementia' ) ]['FolderName'],columns=['FolderName'])
dementia_df['class'] = 1
nondementia_df = pd.DataFrame(study_criteria_table[(criteria_df['Predicted_Stage']== 'No Dementia' )]['FolderName'],columns=['FolderName']).sample(1000)
nondementia_df['class'] = 0

In [ ]:
print('Dementia:',len(dementia_df))
print('No Dementia:',len(nondementia_df))

In [ ]:
y = pd.concat([nondementia_df,dementia_df])
y = y.sample(frac=1)
y=y.reset_index(drop=True)
X = y.merge(data_df,on=['FolderName'],how='left')
X = X[X.columns[3:]]
y = y[y.columns[1]]

In [ ]:
X.to_csv('X_DM_CN_V2.csv',index=False)
y.to_csv('y_DM_CN_V2.csv',index=False)

In [ ]:
def isNaN(num):
    return num != num

In [ ]:
#X.to_csv('X_DM_CN.csv',index=False)
#y.to_csv('y_DM_CN.csv',index=False)
#X = pd.read_csv('X_DM_CN.csv')
#y = pd.read_csv('y_DM_CN.csv')

# Logistic Regression

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0],[0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.array([])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    skb = SelectKBest(f_classif,k=350)
    fs_model = RandomForestClassifier(random_state=7,class_weight='balanced')
    model = LogisticRegression(random_state=7,penalty='elasticnet',solver='saga',class_weight='balanced')
    pipeline = Pipeline(
    [ ("filter",skb),
        ("feature_selection", SelectFromModel(fs_model)),
        ("classification",model)])
    
    params = { 'feature_selection__threshold':[0.00005,0.0001,0.0005,0.001,0.002,0.003,0.004,0.005,0.007,0.01],
        'classification__C': [10**x for x in range(-3,5)],
        'classification__l1_ratio' : [0.5,0.6,0.7,0.8,0.9]}
    
     
    gd_search = GridSearchCV(pipeline, params, scoring='roc_auc', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.predict_proba(X_test)[:,1]
    y_pred =  best_model.predict(X_test)
    auc = metrics.roc_auc_score(y_test, y_prob)
    f1 = metrics.f1_score(y_test, y_pred)
    
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS F1:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred)))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred)))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
with open('DM_v_CN_LG_scores_V2.pickle', 'wb') as f:
    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)
#with open('DM_v_CN_LG_scores_V2.pickle', 'rb') as f:
#    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)

In [ ]:
cm_total

In [ ]:
plot_confusion_matrix(cm=cm_total,
                          target_names=['Nondementia','Dementia'],
                          title='Confusion matrix',
                          normalize=True)
report= metrics.classification_report(y_test_all,y_pred_all)

specificity = cm_total[1,1]/(cm_total[1,0]+cm_total[1,1])
sensitivity = cm_total[0,0]/(cm_total[0,0]+cm_total[0,1])
f1 = metrics.f1_score(y_test_all, y_pred_all)

print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)
print('F1 :',f1)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)

print(report)

In [ ]:
#Maximize F1 & PR-ROC
cm=cm_total
thresholds = np.arange(0.1, 1, 0.02).tolist()
for threshold in thresholds:
    print("threshold:",threshold)
    y_pred = (y_prob_all >= threshold).astype(int)
    score = metrics.accuracy_score(y_test_all,y_pred)
    print("score:",score)
    kappa = metrics.cohen_kappa_score(y_test_all,y_pred)
    print("kappa:",kappa)
    f1 = metrics.f1_score(y_test_all,y_pred)
    print("F-measure:",f1)
    f1_weighted = metrics.f1_score(y_test_all,y_pred,average='weighted')
    print("F1 Weighted:",f1_weighted)
    accuracy = metrics.accuracy_score(y_test_all,y_pred)
    print('Accuracy: %f' % accuracy)
    precision = metrics.precision_score(y_test_all,y_pred)
    print('Precision: %f' % precision)
    recall = metrics.recall_score(y_test_all,y_pred)
    print('Recall: %f' % recall)
    
    specificity = cm[1,1]/(cm[1,0]+cm[1,1])
    sensitivity = cm[0,0]/(cm[0,0]+cm[0,1])
    print('Sensitivity : ', sensitivity )
    print('Specificity : ', specificity)
    print("\n")
    cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[0, 1])
    plot_confusion_matrix(cm=cm,
                              target_names=['Nondementia','Dementia'],
                              title='Confusion matrix',
                              cmap=None,
                              normalize=True)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
roc_auc = metrics.auc(fpr, tpr)
plt.figure(dpi=300)
plt.plot(fpr, tpr, color='darkorange', label='AUROC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
plt.figure(dpi=300)
average_precision = average_precision_score(y_test_all, y_prob_all)
precision, recall, thresholds = precision_recall_curve(y_test_all, y_prob_all)

plt.plot(recall, precision,color='darkorange', label='AUPRC = %0.2f)' % average_precision)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.legend(loc="upper right")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.title('Precision-Recall curve'.format(
          average_precision))

In [ ]:
#Final Model
scaler = preprocessing.StandardScaler().fit(X)
X_train = pd.DataFrame(data=scaler.transform(X),columns=X.columns)
imputer = KNNImputer(n_neighbors=10).fit(X_train)
X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
features = np.array(X.columns)
skb = SelectKBest(f_classif,k=350)
skb.fit(X_train, y)
features = features[skb.get_support()]
X_train= X_train[features] 
fs_model = SelectFromModel(RandomForestClassifier(random_state=7,class_weight='balanced'),threshold=0.003)
fs_model.fit(X_train,y)
features = features[fs_model.get_support()]
X_train= X_train[features]
final_model = LogisticRegression(random_state=7,C = 0.01 ,penalty='elasticnet',solver='saga',l1_ratio=0.6,class_weight = 'balanced')
final_model.fit(X_train,y)


In [ ]:
with open('DM_v_CN_LG_model.pickle', 'wb') as f:
     pickle.dump([final_model], f)
#with open('DM_v_CN_LG_scores.pickle', 'rb') as f:
#    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)


In [ ]:
target_names=['Nondementia','Dementia']
feat_with_weights = sorted(zip(final_model.coef_[0], list(X_train.columns)))
plt.figure(figsize=(20,10)) 
x_values = [r[1]for r in feat_with_weights[:20]]
y_values = [r[0]for r in feat_with_weights[:20]]
x_values.reverse()
y_values.reverse()
plt.barh(x_values,y_values)
plt.xticks(rotation='80')
plt.xlabel('Feature Weight')
plt.title('Important Features for ' + target_names[0])
plt.show()

In [ ]:
feat_with_weights = sorted(zip(final_model.coef_[0], list(X_train.columns)))
plt.figure(figsize=(20,10)) 
x_values = [r[1]for r in feat_with_weights[-20:]]
y_values = [r[0]for r in feat_with_weights[-20:]]
plt.barh(x_values,y_values)
plt.xticks(rotation='80')
plt.xlabel('Feature Weight')
plt.title('Important Features for ' + target_names[1])
plt.show()

# SVM

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0],[0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.array([])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):
    f+=1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    #imputer after scaler
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the model
    skb = SelectKBest(f_classif,k=350)
    fs_model = RandomForestClassifier(random_state=7,class_weight='balanced')   
    model = SGDClassifier(loss='hinge',random_state=7,class_weight='balanced')
    pipeline = Pipeline(
    [ ("filter",skb),
        ("feature_selection", SelectFromModel(fs_model)),
        ("classification",model)])
    params = {  'feature_selection__threshold':[0.00005,0.0001,0.0005,0.001,0.002,0.003,0.004,0.005,0.007,0.01],'classification__alpha': [10**x for x in range(-5,5)]}

     
    gd_search = GridSearchCV(pipeline, params, scoring='roc_auc', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.decision_function(X_test)
    y_pred =  best_model.predict(X_test)
    auc = metrics.roc_auc_score(y_test, y_prob)
    f1 = metrics.f1_score(y_test, y_pred)
    
    
    # store the result
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS F1:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred)))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred)))
    print('F1 Score : ' + str(f1))
    kappa = metrics.cohen_kappa_score(y_test,y_pred)
    print('Kappa : ' + str(kappa))
    kappa_all.append(kappa)
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model
print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
#with open('DM_v_CN_SVM_scores.pickle', 'wb') as f:
#    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)
with open('DM_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)

In [ ]:
plot_confusion_matrix(cm=cm_total,
                          target_names=['Nondementia','Dementia'],
                          title='Confusion matrix',
                          normalize=True)
report= metrics.classification_report(y_test_all,y_pred_all)

specificity = cm_total[1,1]/(cm_total[1,0]+cm_total[1,1])
sensitivity = cm_total[0,0]/(cm_total[0,0]+cm_total[0,1])
f1 = metrics.f1_score(y_test_all, y_pred_all)

print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)
print('F1 :',f1)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)

print(report)

In [ ]:
#Maximize F1 & PR-ROC
cm=cm_total
thresholds = np.arange(0.1, 1, 0.02).tolist()
for threshold in thresholds:
    print("threshold:",threshold)
    y_pred = (y_prob_all >= threshold).astype(int)
    score = metrics.accuracy_score(y_test_all,y_pred)
    print("score:",score)
    kappa = metrics.cohen_kappa_score(y_test_all,y_pred)
    print("kappa:",kappa)
    f1 = metrics.f1_score(y_test_all,y_pred)
    print("F-measure:",f1)
    f1_weighted = metrics.f1_score(y_test_all,y_pred,average='weighted')
    print("F1 Weighted:",f1_weighted)
    accuracy = metrics.accuracy_score(y_test_all,y_pred)
    print('Accuracy: %f' % accuracy)
    precision = metrics.precision_score(y_test_all,y_pred)
    print('Precision: %f' % precision)
    recall = metrics.recall_score(y_test_all,y_pred)
    print('Recall: %f' % recall)
    
    specificity = cm[1,1]/(cm[1,0]+cm[1,1])
    sensitivity = cm[0,0]/(cm[0,0]+cm[0,1])
    print('Sensitivity : ', sensitivity )
    print('Specificity : ', specificity)
    print("\n")
    cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[0, 1])
    plot_confusion_matrix(cm=cm,
                              target_names=['Nondementia','Dementia'],
                              title='Confusion matrix',
                              cmap=None,
                              normalize=True)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
roc_auc = metrics.auc(fpr, tpr)
plt.figure(dpi=300)
plt.plot(fpr, tpr, color='darkorange', label='AUROC = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend(loc="lower right")
plt.show()

In [ ]:
y_prob_all

In [ ]:
plt.figure(dpi=300)
average_precision = average_precision_score(y_test_all, y_prob_all)
precision, recall, thresholds = precision_recall_curve(y_test_all, y_prob_all)

plt.plot(recall, precision,color='darkorange', label='AUPRC = %0.2f)' % average_precision)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.legend(loc="upper right")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.title('Precision-Recall curve'.format(
          average_precision))

# Random Forest

In [ ]:
#Include Random Forest Model https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.SelectFromModel.html

In [ ]:
# configure the cross-validation procedure
cv_outer = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)

# enumerate splits
cm_total = [[0,0],[0,0]]
auc_all = list()
kappa_all = list()
f1_all = list()
f= 0
y_test_all =np.array([])
y_prob_all =np.array([])
y_pred_all =np.array([])
for train_idx, test_idx in cv_outer.split(X,y):#TODO X, y
    f+= 1
    # split data
    print('Computing Fold ' + str(f))
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    
    
    scaler = preprocessing.StandardScaler().fit(X_train)
    X_train = pd.DataFrame(data=scaler.transform(X_train),columns=X_train.columns)
    imputer = KNNImputer(n_neighbors=10).fit(X_train)
    X_train = pd.DataFrame(data=imputer.transform(X_train),columns=X_train.columns)
    
    X_test = pd.DataFrame(data=scaler.transform(X_test),columns=X_train.columns)
    X_test = pd.DataFrame(data=imputer.transform(X_test),columns=X_train.columns)
    
    # configure the cross-validation procedure
    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    
    # define the pipeline
    model = RandomForestClassifier(random_state=7,class_weight='balanced')
    pipeline = Pipeline(
    [ ("feature_selection", SelectFromModel(model)),
        ("classification",model)])
    
    params = { 'feature_selection__threshold':[0.00005,0.0001,0.0005,0.001,0.005,0.01],
        'classification__max_depth': [3, 5, 7, 10],
        'classification__n_estimators' : [50, 100, 200],
        'classification__ccp_alpha': [0.01, 0.2,0.4,0.5, 0.1]}
    gd_search = GridSearchCV(pipeline, params, scoring='roc_auc', n_jobs=-1, cv=cv_inner).fit(X_train, y_train)
    best_params = gd_search.best_params_
    best_model = gd_search.best_estimator_

    y_prob = best_model.predict_proba(X_test)[:,1]
    y_pred =  best_model.predict(X_test)
    auc = metrics.roc_auc_score(y_test, y_prob)
    f1 = metrics.f1_score(y_test, y_pred)
    kappa = metrics.cohen_kappa_score(y_test,y_pred)

    # store the result
    kappa_all.append(kappa)
    auc_all.append(auc)
    f1_all.append(f1)
    y_test_all = np.concatenate((y_test_all, y_test))
    y_pred_all = np.concatenate((y_pred_all, y_pred))
    y_prob_all = np.concatenate((y_prob_all, y_prob))
    
    print("Val Auc:",auc, "Best GS Auc:",gd_search.best_score_, "Best Params:",gd_search.best_params_)
    print('Accuracy Score : ' + str(metrics.accuracy_score(y_test, y_pred)))
    print('Precision Score : ' + str(metrics.precision_score(y_test, y_pred)))
    print('Recall Score : ' + str(metrics.recall_score(y_test, y_pred)))
    print('F1 Score : ' + str(metrics.f1_score(y_test, y_pred)))
    print('Kappa : ' + str(kappa))
    
    cm = metrics.confusion_matrix(y_test,y_pred,labels=[0, 1])
    cm_total = cm + cm_total
    
# summarize the estimated performance of the model

print('Estimated AUC: %.3f (%.3f)' % (np.mean(auc_all), np.std(auc_all)))
print('Estimated Kappa: %.3f (%.3f)' % (np.mean(kappa_all), np.std(kappa_all)))
print('Estimated F1: %.3f (%.3f)' % (np.mean(f1_all), np.std(f1_all)))

In [ ]:
with open('DM_v_CN_RF_scores.pickle', 'wb') as f:
    pickle.dump([y_test_all,y_prob_all,y_pred_all,cm_total], f)
#with open('DM_v_CN_RF_scores.pickle', 'rb') as f:
#    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)

In [ ]:
plot_confusion_matrix(cm=cm_total,
                          target_names=['Nondementia','Dementia'],
                          title='Confusion matrix',
                          normalize=True)
report= metrics.classification_report(y_test_all,y_pred_all)

specificity = cm_total[1,1]/(cm_total[1,0]+cm_total[1,1])
sensitivity = cm_total[0,0]/(cm_total[0,0]+cm_total[0,1])
f1 = metrics.f1_score(y_test_all, y_pred_all)

print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)
print('F1 :',f1)
f1_weighted = metrics.f1_score(y_test_all,y_pred_all,average='weighted')
print("F1 Weighted:",f1_weighted)

print(report)

In [ ]:
#Maximize F1 & PR-ROC
cm=cm_total
thresholds = np.arange(0.1, 1, 0.02).tolist()
for threshold in thresholds:
    print("threshold:",threshold)
    y_pred = (y_prob_all >= threshold).astype(int)
    score = metrics.accuracy_score(y_test_all,y_pred)
    print("score:",score)
    kappa = metrics.cohen_kappa_score(y_test_all,y_pred)
    print("kappa:",kappa)
    f1 = metrics.f1_score(y_test_all,y_pred)
    print("F-measure:",f1)
    f1_weighted = metrics.f1_score(y_test_all,y_pred,average='weighted')
    print("F1 Weighted:",f1_weighted)
    accuracy = metrics.accuracy_score(y_test_all,y_pred)
    print('Accuracy: %f' % accuracy)
    precision = metrics.precision_score(y_test_all,y_pred)
    print('Precision: %f' % precision)
    recall = metrics.recall_score(y_test_all,y_pred)
    print('Recall: %f' % recall)
    
    specificity = cm[1,1]/(cm[1,0]+cm[1,1])
    sensitivity = cm[0,0]/(cm[0,0]+cm[0,1])
    print('Sensitivity : ', sensitivity )
    print('Specificity : ', specificity)
    print("\n")
    cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[0, 1])
    plot_confusion_matrix(cm=cm,
                              target_names=['Nondementia','Dementia'],
                              title='Confusion matrix',
                              cmap=None,
                              normalize=True)

In [ ]:
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
roc_auc = metrics.auc(fpr, tpr)
plt.figure(dpi=300)
plt.plot(fpr, tpr, color='darkorange', label='ROC curve (area = %0.2f)' % roc_auc)
plt.plot([0, 1], [0, 1], color='navy', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
#plt.title('ROC Curve for Binary Classification of Dementia using Random Forest')
plt.legend(loc="lower right")
plt.show()

In [ ]:
plt.figure(dpi=300)
average_precision = average_precision_score(y_test_all, y_prob_all)
precision, recall, thresholds = precision_recall_curve(y_test_all, y_prob_all)

plt.plot( recall, precision,color='darkorange', label='AUPRC=%0.2f)' % average_precision)
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.legend(loc="upper right")
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.05])
plt.xlim([0.0, 1.0])
plt.title('Precision-Recall curve'.format(
          average_precision))

In [ ]:
#Final Model

scaler = preprocessing.StandardScaler().fit(X)
X_processed = pd.DataFrame(data=imputer.transform(X),columns=X.columns)
imputer = KNNImputer(n_neighbors=10).fit(X_processed)
X_processed = pd.DataFrame(data=scaler.transform(X_processed),columns=X.columns)

final_model = RandomForestClassifier(random_state=7,ccp_alpha=0.01,max_depth= 10, n_estimators=200)
final_model = SelectFromModel(final_model,threshold=0.0001)
final_model.fit(X_processed,y)

In [ ]:
#with open('DM_v_CN_RF_model.pickle', 'wb') as f:
#    pickle.dump([final_model], f)
#with open('DM_v_CN_RF_scores.pickle', 'rb') as f:
#    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)


In [ ]:
feature_importance_df = pd.DataFrame(columns=['feature','importance'])
feature_importance_df['feature'] = X.columns
feature_importance_df['importance'] = final_model.estimator_.feature_importances_
feature_importance_df
feature_importance_df.sort_values(by=['importance'],ascending=False)[:50]

# Figure

In [ ]:
plt.figure(figsize=(4,3),dpi=300)
with open('DM_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
roc_auc = metrics.auc(fpr, tpr)

plt.plot(fpr, tpr, color='tab:blue', label='LR AUROC=%0.2f' % roc_auc)

with open('DM_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
roc_auc = metrics.auc(fpr, tpr)
plt.plot(fpr, tpr, color='tab:red', label='SVM AUROC=%0.2f' % roc_auc)

with open('DM_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
roc_auc = metrics.auc(fpr, tpr)
plt.plot(fpr, tpr, color='green', label='RF AUROC=%0.2f' % roc_auc)


plt.plot([0, 1], [0, 1], color='black', linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.legend(loc="lower right",frameon=False)
plt.title('ROC Curve')
plt.savefig('../figures/Fig5_A.svg',format='svg')
plt.show()


In [ ]:
plt.figure(figsize=(4,3),dpi=300)
with open('DM_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
average_precision = average_precision_score(y_test_all, y_prob_all)
precision, recall, thresholds = precision_recall_curve(y_test_all, y_prob_all)
plt.plot( recall, precision,color='tab:blue', label='LG AUPRC=%0.2f' % average_precision)

with open('DM_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
average_precision = average_precision_score(y_test_all, y_prob_all)
precision, recall, thresholds = precision_recall_curve(y_test_all, y_prob_all)
plt.plot( recall, precision,color='tab:red', label='SVM AUPRC=%0.2f' % average_precision)

with open('DM_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
average_precision = average_precision_score(y_test_all, y_prob_all)
precision, recall, thresholds = precision_recall_curve(y_test_all, y_prob_all)
plt.plot( recall, precision,color='green', label='RF AUPRC=%0.2f' % average_precision)
baseline = len(y_test_all[y_test_all==1]) / len(y_test_all)
plt.plot([0, 1], [baseline, baseline], linestyle='--', color='black')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.0])
plt.legend(loc="upper right",frameon=False)
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.ylim([0.0, 1.0])
plt.xlim([0.0, 1.0])
plt.title('Precision-Recall curve'.format(
          average_precision))
plt.savefig('../figures/Fig5_B.svg',format='svg')
plt.show()

In [ ]:
#Maximize F1 & PR-ROC

with open('DM_v_CN_SVM_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
    
#cm=cm_total
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
#optimal_idx = np.argmax(tpr - fpr)
#threshold = thresholds[optimal_idx]
threshold = -0.41
print("threshold:",threshold)
y_pred = (y_prob_all >= threshold).astype(int)
score = metrics.accuracy_score(y_test_all,y_pred)
print("score:",score)
kappa = metrics.cohen_kappa_score(y_test_all,y_pred)
print("kappa:",kappa)
f1 = metrics.f1_score(y_test_all,y_pred)
print("F-measure:",f1)
f1_micro = metrics.f1_score(y_test_all,y_pred,average='micro')
print("F1 micro:",f1_micro)
f1_macro = metrics.f1_score(y_test_all,y_pred,average='macro')
print("F1 macro:",f1_macro)
f1_weighted = metrics.f1_score(y_test_all,y_pred,average='weighted')
print("F1 Weighted:",f1_weighted)
accuracy = metrics.accuracy_score(y_test_all,y_pred)
print('Accuracy: %f' % accuracy)
precision = metrics.precision_score(y_test_all,y_pred)
print('Precision: %f' % precision)
recall = metrics.recall_score(y_test_all,y_pred)
print('Recall: %f' % recall)


cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[ 1,0])
specificity = cm[1,1]/(cm[1,0]+cm[1,1])
sensitivity = cm[0,0]/(cm[0,0]+cm[0,1])
print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)
print("\n")
target_names=['DEM','CN']
title='Confusion matrix'
cmap=None
normalize=True

accuracy = np.trace(cm) / float(np.sum(cm))
misclass = 1 - accuracy

if cmap is None:
    cmap = plt.get_cmap('Blues')
    newcolors = np.vstack((cmap(np.linspace(0, 0.01, 60)),cmap(np.linspace(0, 1, 128)),
    cmap(np.linspace(0.99, 1, 50))))
    cmap = ListedColormap(newcolors)

if normalize:
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(5, 4),dpi=300)
plt.imshow(cm, interpolation='nearest',vmin=0.0,vmax=1, cmap=cmap)
#plt.title(title)
plt.colorbar()

if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=0)
    plt.yticks(tick_marks, target_names)


thresh = cm.max() / 1.5 if normalize else cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if normalize:
        plt.text(j, i, "{:0.3f}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    else:
        plt.text(j, i, "{:,}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")


plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.savefig('../figures/Fig5_C.svg',format='svg')

In [ ]:
#Maximize F1 & PR-ROC

with open('DM_v_CN_LG_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
    
#cm=cm_total
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
#optimal_idx = np.argmax(tpr - fpr)
#threshold = thresholds[optimal_idx]
threshold = 0.4
print("threshold:",threshold)
y_pred = (y_prob_all >= threshold).astype(int)
score = metrics.accuracy_score(y_test_all,y_pred)
print("score:",score)
kappa = metrics.cohen_kappa_score(y_test_all,y_pred)
print("kappa:",kappa)
f1 = metrics.f1_score(y_test_all,y_pred)
print("F-measure:",f1)
f1_micro = metrics.f1_score(y_test_all,y_pred,average='micro')
print("F1 micro:",f1_micro)
f1_macro = metrics.f1_score(y_test_all,y_pred,average='macro')
print("F1 macro:",f1_macro)
f1_weighted = metrics.f1_score(y_test_all,y_pred,average='weighted')
print("F1 Weighted:",f1_weighted)
accuracy = metrics.accuracy_score(y_test_all,y_pred)
print('Accuracy: %f' % accuracy)
precision = metrics.precision_score(y_test_all,y_pred)
print('Precision: %f' % precision)
recall = metrics.recall_score(y_test_all,y_pred)
print('Recall: %f' % recall)


cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[ 1,0])
specificity = cm[1,1]/(cm[1,0]+cm[1,1])
sensitivity = cm[0,0]/(cm[0,0]+cm[0,1])
print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)
print("\n")
target_names=['DM','CN']
title='Confusion matrix'
cmap=None
normalize=True

accuracy = np.trace(cm) / float(np.sum(cm))
misclass = 1 - accuracy

if cmap is None:
    cmap = plt.get_cmap('Blues')
    #bottom = plt.get_cmap('Blues', 128)
    #top = plt.get_cmap('Blues', 128)

    newcolors = np.vstack((cmap(np.linspace(0, 0.01, 50)),cmap(np.linspace(0, 1, 128)),
    cmap(np.linspace(0.99, 1, 40))))
    cmap = ListedColormap(newcolors)
    
    


if normalize:
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6),dpi=300)
plt.imshow(cm, interpolation='nearest',vmin=0,vmax=1, cmap=cmap)
plt.title(title)
plt.colorbar()

if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=0)
    plt.yticks(tick_marks, target_names)


thresh = cm.max() / 1.5 if normalize else cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if normalize:
        plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    else:
        plt.text(j, i, "{:,}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")


plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label\n\nAccuracy={:0.4f}; Weighted F1={:0.4f}'.format(accuracy, f1_weighted))
plt.savefig('../figures/Fig5_C.svg',format='svg')

In [ ]:
#Maximize F1 & PR-ROC

with open('DM_v_CN_RF_scores.pickle', 'rb') as f:
    y_test_all,y_prob_all,y_pred_all,cm_total = pickle.load(f)
    
#cm=cm_total
fpr, tpr, thresholds = metrics.roc_curve(y_test_all, y_prob_all) 
#optimal_idx = np.argmax(tpr - fpr)
#threshold = thresholds[optimal_idx]
threshold = 0.4
print("threshold:",threshold)
y_pred = (y_prob_all >= threshold).astype(int)
score = metrics.accuracy_score(y_test_all,y_pred)
print("score:",score)
kappa = metrics.cohen_kappa_score(y_test_all,y_pred)
print("kappa:",kappa)
f1 = metrics.f1_score(y_test_all,y_pred)
print("F-measure:",f1)
f1_micro = metrics.f1_score(y_test_all,y_pred,average='micro')
print("F1 micro:",f1_micro)
f1_macro = metrics.f1_score(y_test_all,y_pred,average='macro')
print("F1 macro:",f1_macro)
f1_weighted = metrics.f1_score(y_test_all,y_pred,average='weighted')
print("F1 Weighted:",f1_weighted)
accuracy = metrics.accuracy_score(y_test_all,y_pred)
print('Accuracy: %f' % accuracy)
precision = metrics.precision_score(y_test_all,y_pred)
print('Precision: %f' % precision)
recall = metrics.recall_score(y_test_all,y_pred)
print('Recall: %f' % recall)


cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[ 1,0])
specificity = cm[1,1]/(cm[1,0]+cm[1,1])
sensitivity = cm[0,0]/(cm[0,0]+cm[0,1])
print('Sensitivity : ', sensitivity )
print('Specificity : ', specificity)
print("\n")
target_names=['DM','CN']
title='Confusion matrix'
cmap=None
normalize=True

accuracy = np.trace(cm) / float(np.sum(cm))
misclass = 1 - accuracy

if cmap is None:
    cmap = plt.get_cmap('Blues')
    newcolors = np.vstack((cmap(np.linspace(0, 0.01, 60)),cmap(np.linspace(0, 1, 128)),
    cmap(np.linspace(0.99, 1, 50))))
    cmap = ListedColormap(newcolors)

if normalize:
    cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

plt.figure(figsize=(8, 6),dpi=300)
plt.imshow(cm, interpolation='nearest',vmin=0,vmax=1, cmap=cmap)
plt.title(title)
plt.colorbar()

if target_names is not None:
    tick_marks = np.arange(len(target_names))
    plt.xticks(tick_marks, target_names, rotation=0)
    plt.yticks(tick_marks, target_names)


thresh = cm.max() / 1.5 if normalize else cm.max() / 2
for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
    if normalize:
        plt.text(j, i, "{:0.4f}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")
    else:
        plt.text(j, i, "{:,}".format(cm[i, j]),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")


plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label\n\nAccuracy={:0.4f}; Weighted F1={:0.4f}'.format(accuracy, f1_weighted))
plt.savefig('../figures/Fig5_C.svg',format='svg')

In [ ]:
cm = metrics.confusion_matrix(y_test_all,y_pred,labels=[ 1,0])


In [ ]:
cm